# When Sources Disagree on Speaker Counts

CLDR, CIA, and LinguaMeta don't always agree on how many people speak a language
in a given country. This notebook compares per-country speaker estimates across
sources using `db.speaker_counts`.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [1]:
%pip install -q pandas plotly matplotlib



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [3]:
db = low.LanguagesOfTheWorld()
print(db)
for source in ("cldr", "cia", "linguameta", "scraped"):
    print(f"  {source}: {len(db.speaker_counts.by_source(source)):,} records")

LanguagesOfTheWorld(languages=7929, countries=247, continents=5, scripts=106, speaker_counts=8969, language_names=167917)
  cldr: 1,576 records
  cia: 86 records
  linguameta: 5,955 records
  scraped: 1,352 records


## 2 · Query by source, country, and language

In [4]:
rw = db.countries.get("RW")
print(f"{rw.label} population: {rw.population:,}")
print()
for sc in rw.speaker_counts:
    print(f"  {sc.language.label}: {sc.speaker_count:,} ({sc.speaker_fraction:.1%}) [{sc.source}]")

print()
kin = db.languages.get("kin")
print(f"{kin.label} — per-country breakdown:")
for sc in kin.speaker_counts:
    print(f"  {sc.country.label}: {sc.speaker_count:,} ({sc.source})")

Rwanda population: 13,623,300

  English: 2,043,495 (15.0%) [cldr]
  French: 790,151 (5.8%) [cldr]
  Kinyarwanda: 10,489,941 (77.0%) [cldr]
  English: 1,900,000 (13.9%) [linguameta]
  French: 2,300 (0.0%) [linguameta]
  Kinyarwanda: 9,800,000 (71.9%) [linguameta]

Kinyarwanda — per-country breakdown:
  Congo, Democratic Republic of the: 438,531 (cldr)
  Rwanda: 10,489,941 (cldr)
  Rwanda: 9,800,000 (linguameta)
  Uganda: 1,034,943 (cldr)


## 3 · Build a comparison table for CLDR vs CIA

Pair records that share the same `(country, language)` key.

In [5]:
def build_pairs(source_a, source_b):
    lookup = {}
    for sc in db.speaker_counts.by_source(source_a):
        key = (sc.country.code, sc.language.part3)
        lookup.setdefault(key, {})[source_a] = sc
    rows = []
    for sc in db.speaker_counts.by_source(source_b):
        key = (sc.country.code, sc.language.part3)
        if key not in lookup:
            continue
        a = lookup[key][source_a]
        rows.append({
            "country": sc.country.label,
            "country_code": sc.country.code,
            "language": sc.language.label,
            "part3": sc.language.part3,
            f"{source_a}_count": a.speaker_count,
            f"{source_a}_fraction": a.speaker_fraction,
            f"{source_b}_count": sc.speaker_count,
            f"{source_b}_fraction": sc.speaker_fraction,
            "population": sc.country.population,
        })
    return pd.DataFrame(rows)

pairs_df = build_pairs("cldr", "cia")
print(f"Country-language pairs in both CLDR and CIA: {len(pairs_df)}")
pairs_df.head(10)

Country-language pairs in both CLDR and CIA: 45


,country,country_code,language,part3,cldr_count,cldr_fraction,cia_count,cia_fraction,population
0,Andorra,AD,French,fra,5805,0.068,8537,0.100,85370
1,Angola,AO,Kimbundu,kmb,9300525,0.250,3040814,0.078,37202100
2,Angola,AO,Portuguese,por,24925407,0.670,27757174,0.712,37202100
3,Angola,AO,Umbundu,umb,10788609,0.290,8966503,0.230,37202100
4,American Samoa,AS,English,eng,42578,0.970,1427,0.033,43895
5,American Samoa,AS,Samoan,smo,43456,0.990,38032,0.879,43895
6,Australia,AU,English,eng,25697856,0.960,19793463,0.720,26768600
7,Burkina Faso,BF,Dyula,dyu,7373504,0.320,1338947,0.057,23042200
8,Burkina Faso,BF,Mossi,mos,9216880,0.400,12426368,0.529,23042200
9,Estonia,EE,Russian,rus,668522,0.560,382036,0.285,1193790


## 4 · Scatter plot — CLDR vs CIA counts

In [6]:
fig = px.scatter(
    pairs_df,
    x="cldr_count",
    y="cia_count",
    hover_name="language",
    hover_data={"country": True, "cldr_count": ":,", "cia_count": ":,"},
    log_x=True,
    log_y=True,
    labels={"cldr_count": "CLDR speaker count", "cia_count": "CIA speaker count"},
    title="CLDR vs CIA Speaker Counts (log-log)",
)
max_val = max(pairs_df["cldr_count"].max(), pairs_df["cia_count"].max())
fig.add_shape(type="line", x0=1, y0=1, x1=max_val, y1=max_val,
              line=dict(dash="dash", color="gray"))
fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 5 · Relative disagreement

In [7]:
pairs_df["ratio"] = pairs_df.apply(
    lambda r: max(r["cldr_count"], r["cia_count"]) / max(min(r["cldr_count"], r["cia_count"]), 1),
    axis=1,
)

fig2 = px.histogram(
    pairs_df,
    x="ratio",
    nbins=30,
    labels={"ratio": "Max/min count ratio"},
    title="Distribution of CLDR/CIA Disagreement (ratio > 1 means sources differ)",
)
fig2.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig2.show()

print("Largest disagreements:")
print(pairs_df.nlargest(10, "ratio")[["country", "language", "cldr_count", "cia_count", "ratio"]].to_string(index=False))

Largest disagreements:
              country  language  cldr_count  cia_count     ratio
                Nauru   English        9397        198 47.459596
       American Samoa   English       42578       1427 29.837421
            Mauritius    French      956665      53766 17.793122
           Luxembourg   English      375902      24165 15.555638
              Namibia Afrikaans     2102745     268161  7.841353
         Burkina Faso     Dyula     7373504    1338947  5.506942
              Liberia   English     4512917    1112708  4.055796
                Samoa   English        4385      14084  3.211859
               Angola  Kimbundu     9300525    3040814  3.058564
Sao Tome and Principe    French       44712      15202  2.941192


## 6 · Collection helpers

In [8]:
de_entries = db.speaker_counts.for_country("DE")
print(f"Speaker count records for Germany: {len(de_entries)}")
for sc in de_entries[:6]:
    print(f"  {sc.language.label}: {sc.speaker_count:,} ({sc.source})")

print()
deutsch = db.speaker_counts.for_language("deu")
print(f"Speaker count records for German worldwide: {len(deutsch)}")
for sc in deutsch[:6]:
    print(f"  {sc.country.label}: {sc.speaker_count:,} ({sc.source})")

Speaker count records for Germany: 41
  Bavarian: 14,300,247 (cldr)
  Danish: 1,682,382 (cldr)
  German: 76,548,381 (cldr)
  Lower Sorbian: 6,981 (cldr)
  Modern Greek (1453-): 319,652 (cldr)
  English: 53,836,224 (cldr)

Speaker count records for German worldwide: 35
  Austria: 8,698,940 (cldr)
  Austria: 8,600,000 (linguameta)
  Belgium: 2,635,072 (cldr)
  Belgium: 2,600,000 (linguameta)
  Bulgaria: 542,612 (cldr)
  Brazil: 1,848,436 (cldr)


## 7 · Summary

In [9]:
print(f"Pairs compared: {len(pairs_df)}")
print(f"Median disagreement ratio: {pairs_df['ratio'].median():.2f}")
print(f"Pairs where ratio > 2: {(pairs_df['ratio'] > 2).sum()}")

Pairs compared: 45
Median disagreement ratio: 1.38
Pairs where ratio > 2: 15
